In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/schools_with_athletics.csv")
print("Loaded:", df.shape)

# Collapse 12 city-size labels into 4 groups clients understand
def city_group(s):
    if pd.isna(s):
        return None
    s = s.lower()
    for k in ["city", "suburb", "town", "rural"]:
        if k in s:
            return k.capitalize()
df["city_group"] = df["city_size"].apply(city_group)

Loaded: (3147, 44)


In [2]:
# One client = one intake conversation, stored as a dictionary.
client = {
    "name": "Test client: Brazilian male soccer player",
    "max_budget": 25000,          # max sticker cost per year (USD), before scholarships
    "budget_flex": 1.5,           # athletes may get aid, so allow sticker up to 1.5x budget
    "school_types": ["2-year", "4-year"],
    "states": None,               # e.g. ["KS", "MO", "TX"]; None = anywhere
    "city_groups": None,          # subset of ["City", "Suburb", "Town", "Rural"]; None = any
    "sport": "Soccer",
    "gender": "men",
    "needs_athletic_scholarship": True,
    "min_grad_rate_4yr": 0.30,    # drop 4-year schools below this (unknown grad rate is kept)
    "min_grad_rate_2yr": 0.20,    # lower bar: JUCO rates are understated by early transfers
    "weights": {                  # importance: 0 = ignore ... 5 = critical
        "low_cost": 5,
        "grad_rate": 3,
        "sport_culture": 2,             # big-time, famous program
        "athlete_opportunity": 4,       # athletes are a big share of students
        "international_community": 3,   # % international undergrads
        "open_admission": 0,
        "small_school": 1,
    },
}

In [3]:
def match(df, client, top_n=15):
    c = df.copy()
    print("Start:", len(c))

    # --- 1. Hard filters: fail one = excluded. Print counts so the funnel is visible. ---
    c = c[c["school_type"].isin(client["school_types"])]
    print("After school type:", len(c))
    if client["states"]:
        c = c[c["state"].isin(client["states"])]
        print("After states:", len(c))
    if client["city_groups"]:
        c = c[c["city_group"].isin(client["city_groups"])]
        print("After city size:", len(c))
    limit = client["max_budget"] * client.get("budget_flex", 1.0)
    c = c[c["cost_international"] <= limit]            # unknown cost (NaN) is excluded too
    print(f"After cost <= ${limit:,.0f}:", len(c))
    if client.get("sport"):
        col = "mens_sports" if client["gender"] == "men" else "womens_sports"
        has_sport = c[col].fillna("").apply(
            lambda s: client["sport"] in [x.strip() for x in s.split(";")]
        )
        c = c[has_sport]
        print(f"After has {client['gender']}'s {client['sport']}:", len(c))
    if client.get("needs_athletic_scholarship"):
        c = c[c["athletic_aid_tier"].isin(["Athletic scholarships", "Mixed / verify"])]
        print("After athletic scholarships available:", len(c))
    mg4, mg2 = client.get("min_grad_rate_4yr"), client.get("min_grad_rate_2yr")
    if mg4 is not None or mg2 is not None:
        # Each school gets the threshold for its type; missing grad rate is kept, not punished
        threshold = np.where(c["school_type"] == "4-year", mg4 or 0, mg2 or 0)
        c = c[c["grad_rate"].isna() | (c["grad_rate"] >= threshold)]
        print("After minimum grad rate:", len(c))
    if c.empty:
        print("No schools pass the filters.")
        return c

    # --- 2. Features on a 0-1 scale (percentile among remaining schools). Missing = 0.5 neutral. ---
    f = pd.DataFrame(index=c.index)
    f["low_cost"] = 1 - c["cost_international"].rank(pct=True)     # cheaper = higher
    f["grad_rate"] = c["grad_rate"].rank(pct=True)
    f["sport_culture"] = c["sport_culture_pct"] / 100              # already a percentile
    f["athlete_opportunity"] = c["athlete_share"].rank(pct=True)
    f["international_community"] = c["pct_international"].rank(pct=True)
    f["open_admission"] = c["open_admission"].astype(float)
    f["small_school"] = 1 - c["undergrads"].rank(pct=True)          # smaller = higher
    f = f.fillna(0.5)

    # --- 3. Weighted score 0-100 + the 2 features that contributed most ---
    w = pd.Series(client["weights"], dtype=float)
    contrib = f[w.index] * w
    c["match_score"] = (contrib.sum(axis=1) / w.sum() * 100).round(1)
    c["top_reasons"] = contrib.apply(lambda r: ", ".join(r.nlargest(2).index), axis=1)
    return c.sort_values("match_score", ascending=False).head(top_n)

print(client["name"])
results = match(df, client)
show = ["name", "state", "school_type", "association", "cost_international", "grad_rate",
        "pct_international", "athlete_share", "match_score", "top_reasons"]
print()
print(results[show].round(2).to_string(index=False))

Test client: Brazilian male soccer player
Start: 3147
After school type: 3147
After cost <= $37,500: 1767
After has men's Soccer: 606
After athletic scholarships available: 377
After minimum grad rate: 336

                                                  name state school_type association  cost_international  grad_rate  pct_international  athlete_share  match_score                   top_reasons
                                 Western Texas College    TX      2-year        JUCO             16185.0       0.53               0.10           0.57         85.7 low_cost, athlete_opportunity
University of Arkansas Community College Rich Mountain    AR      2-year        JUCO             13672.0       0.55               0.06           0.36         83.1 low_cost, athlete_opportunity
                         Garden City Community College    KS      2-year        JUCO             15701.0       0.50               0.07           0.32         81.6 low_cost, athlete_opportunity
                     

In [4]:
# A. Same client, 4-year schools only (JUCOs dominated the first list because low_cost = 5)
c_4yr = {**client, "name": "Same client, 4-year only", "school_types": ["4-year"]}
print("=" * 80)
print(c_4yr["name"])
r_4yr = match(df, c_4yr, top_n=10)
print(r_4yr[show].round(2).to_string(index=False))

# B. Sensitivity check: all weights equal. If most of the top 15 survive, the ranking is robust.
c_equal = {**client, "name": "Same client, all weights = 1", "weights": {k: 1 for k in client["weights"]}}
print("\n" + "=" * 80)
print(c_equal["name"])
r_equal = match(df, c_equal, top_n=15)
print(r_equal[show].round(2).to_string(index=False))
overlap = set(results["name"]) & set(r_equal["name"])
print(f"\nOverlap with original top 15: {len(overlap)} of 15")

# C. What's hiding in EADA's "Other" division? (Neosho County landed there)
eada = pd.read_excel("../data/raw/eada/instLevel.xlsx",
                     usecols=["unitid", "institution_name", "state_cd", "classification_name", "ClassificationOther"])
other = eada[eada["classification_name"].str.strip() == "Other"]
print("\n" + "=" * 80)
print("EADA 'Other' schools:", len(other))
print("\nClassificationOther values:")
print(other["ClassificationOther"].value_counts(dropna=False).to_string())

Same client, 4-year only
Start: 3147
After school type: 1797
After cost <= $37,500: 628
After has men's Soccer: 242
After athletic scholarships available: 170
After minimum grad rate: 140
                                     name state school_type         association  cost_international  grad_rate  pct_international  athlete_share  match_score                                  top_reasons
                    Goldey-Beacom College    DE      4-year             NCAA D2             26502.0       0.57               0.09           0.42         82.6                low_cost, athlete_opportunity
                 William Carey University    MS      4-year                NAIA             29651.0       0.60               0.06           0.19         75.0                low_cost, athlete_opportunity
                   Delta State University    MS      4-year             NCAA D2             22445.0       0.48               0.05           0.24         73.9                low_cost, athlete_opportunity



EADA 'Other' schools: 62

ClassificationOther values:
ClassificationOther
NJCAA Division I and II                               4
Liga Atletica Interuniversitaria                      4
Liga Atletica Interuniversitaria (LAI)                3
LAI                                                   2
NJCAA DI and DII depending on sport                   1
4 schools, 3  NCAAlll, one with football, 1 USCAA     1
Tribal College University                             1
NJCAA DI and DII, ACHA DII                            1
NJCAA                                                 1
NJCAA Div I and II                                    1
NWAC and NJCAA D-1                                    1
NJCAA Multi-division                                  1
NCAA Division I-FCS with exception of Football        1
NCCAA Division I & NCCAA Division II                  1
Rodeo                                                 1
NJCAAE, WEC                                           1
NJCAA - Multiple Divisions Ba